In [47]:
library(ANCOMBC)
library(phyloseq)
library(dplyr)
library(tidyr)
library(microbiome)
library(MicrobiomeStat)

In [19]:
library(dplyr); library(stringr); library(tidyr); library(ggplot2)

# ============================================================================
## ANCOM-BC Site-stratified Analysis
Runs one ancombc2() model per Site_Cleaned.
Fixed effects: Age_Group, Sex, Region_custom, Geography
Random effect: BioProject
## Requires:

## NOTE ON INPUT SCALE 
 ANCOM-BC2 estimates sampling fractions from *counts*. load_genus_relabund()  
 returns relative abundances, so we rescale each sample by COUNT_SCALE and  
 round to pseudo-counts. If you have the raw count table, point
 load_genus_counts() (or equivalent) at it below and set RELABUND_INPUT<-FALSE;
 that is the statistically preferable route.
# ============================================================================

In [6]:
source("input_helpers.R")

ANCOMBC_CORES <- 1
MIN_PREVALENCE <- 0.1
MAX_SIGNIFICANCE <- 0.25
RELABUND_INPUT <- TRUE     # set FALSE if you swap in a real count table
COUNT_SCALE <- 1e5         # pseudo-count library size when rescaling proportions

# ---------------------------------------------------------------------------
# Helper: split an ancombc2 term (e.g. "Age_GroupUnder 18 years") into the
# variable and the contrasted level, matching on the longest fixed-effect prefix.
# ---------------------------------------------------------------------------
split_term <- function(term, variables) {
  hits <- variables[startsWith(term, variables)]
  if (length(hits) == 0) return(c(term, term))
  var <- hits[which.max(nchar(hits))]
  lev <- substring(term, nchar(var) + 1L)
  if (lev == "") lev <- var
  c(var, lev)
}

# ---------------------------------------------------------------------------
# Helper: ancombc2 returns a wide table (lfc_*, se_*, W_*, p_*, q_*, diff_*,
# passed_ss_*). Melt it into MaAsLin3-style long format.
# ---------------------------------------------------------------------------
tidy_ancombc2 <- function(res, variables) {
  lfc_cols <- grep("^lfc_", colnames(res), value = TRUE)
  lfc_cols <- setdiff(lfc_cols, "lfc_(Intercept)")

  bind_rows(lapply(lfc_cols, function(cn) {
    term <- sub("^lfc_", "", cn)
    grab <- function(prefix) {
      col <- paste0(prefix, term)
      if (col %in% colnames(res)) res[[col]] else rep(NA, nrow(res))
    }
    parts <- split_term(term, variables)
    data.frame(
      feature         = res$taxon,
      metadata        = parts[1],
      value           = parts[2],
      term            = term,
      coef            = res[[cn]],          # natural-log fold change
      stderr          = grab("se_"),
      stat            = grab("W_"),
      pval_individual = grab("p_"),
      qval_individual = grab("q_"),
      diff_abundant   = grab("diff_"),
      passed_ss       = grab("passed_ss_"),
      model           = "ANCOM-BC2",
      stringsAsFactors = FALSE
    )
  }))
}

In [7]:
cat("=== Loading data ===\n")
genus_data <- load_genus_relabund()
metadata <- load_standard_metadata(dedup = TRUE)

sample_ids <- trimws(as.character(metadata$SampleID))
keep_ids <- !is.na(sample_ids) & sample_ids != "" & !duplicated(sample_ids)
metadata <- metadata[keep_ids, , drop = FALSE]
sample_ids <- sample_ids[keep_ids]

common_samples <- intersect(sample_ids, colnames(genus_data))
cat(sprintf("Common samples: %d\n", length(common_samples)))

abund <- genus_data[, common_samples, drop = FALSE]
meta <- metadata[match(common_samples, sample_ids), , drop = FALSE]
rownames(meta) <- common_samples

for (col in c("Age_Group", "Sex", "Region_custom", "BioProject", "Site_Cleaned", "Geography")) {
  if (col %in% colnames(meta)) meta[[col]] <- trimws(as.character(meta[[col]]))
}

valid <- complete.cases(meta[, c("Age_Group", "Sex", "Region_custom", "BioProject", "Site_Cleaned", "Geography")]) &
  meta$Age_Group != "" & meta$Sex != "" & meta$Region_custom != "" &
  meta$BioProject != "" & meta$Site_Cleaned != "" & meta$Geography != "" &
  meta$Age_Group != "NA" & meta$Sex != "NA" & meta$Geography != "Unknown"
meta <- meta[valid, , drop = FALSE]
abund <- abund[, rownames(meta), drop = FALSE]
cat(sprintf("Samples with complete metadata: %d\n", nrow(meta)))

# Convert proportions -> pseudo-counts for ANCOM-BC2's offset model
if (RELABUND_INPUT) {
  col_tot <- colSums(abund, na.rm = TRUE)
  col_tot[col_tot == 0] <- 1
  abund <- round(sweep(abund, 2, col_tot, "/") * COUNT_SCALE)
  cat(sprintf("Converted relative abundance to pseudo-counts (scale = %g)\n", COUNT_SCALE))
}
# storage.mode(abund) <- "numeric"

meta$Age_Group <- factor(meta$Age_Group)
meta$Sex <- factor(meta$Sex)
meta$Region_custom <- factor(meta$Region_custom)
meta$Geography <- factor(meta$Geography)
meta$BioProject <- factor(meta$BioProject)

MIN_N <- 50
site_table <- meta %>%
  group_by(Site_Cleaned) %>%
  summarise(n = n(), n_bioproject = n_distinct(BioProject), .groups = "drop") %>%
  arrange(desc(n)) %>%
  filter(n >= MIN_N) %>%
  filter(Site_Cleaned == "Saliva")

cat("\n=== Site sizes ===\n")
print(site_table)
# write.csv(site_table, "site_stratified_group_sizes_ancombc2.csv", row.names = FALSE)

# out_root <- "site_stratified_results_ancombc2"
# dir.create(out_root, showWarnings = FALSE, recursive = TRUE)

=== Loading data ===
Common samples: 42424
Samples with complete metadata: 23585
Converted relative abundance to pseudo-counts (scale = 100000)

=== Site sizes ===
# A tibble: 1 x 3
  Site_Cleaned     n n_bioproject
  <chr>        <int>        <int>
1 Saliva       14120           85


In [38]:
geo_results <- list()
for (g in levels(droplevels(sal$Geography))) {
  gm <- sal[sal$Geography == g, , drop = FALSE]
  vars <- c("Age_Group","Sex","Region_custom","BioProject")

  ## 1) 先删协变量含 NA 的样本
  gm <- gm[complete.cases(gm[, vars]), , drop = FALSE]

  ## 2) taxa 过滤（含 prv_cut），再删零计数样本
  sub <- abund[, rownames(gm), drop = FALSE]
  sub <- sub[rowSums(sub) > 0, , drop = FALSE]
  sub <- sub[rowSums(sub > 0) >= 2, , drop = FALSE]
  sub <- sub[rowMeans(sub > 0) >= MIN_PREVALENCE, , drop = FALSE]
  if (nrow(sub) < 5) next
  gm  <- gm[colSums(sub) > 0, , drop = FALSE]
  sub <- sub[, rownames(gm), drop = FALSE]

  ## 3) 现在才 droplevels
  for (v in vars) gm[[v]] <- droplevels(gm[[v]])
  if (nlevels(gm$BioProject) < 2) next
  if (nlevels(gm$Region_custom) >= 2 && "V4" %in% levels(gm$Region_custom))
    gm$Region_custom <- relevel(gm$Region_custom, "V4")

  ## 4) 只保留 >=2 水平的固定效应
  fe <- c("Age_Group","Sex","Region_custom")
  fe <- fe[vapply(fe, function(x) nlevels(gm[[x]]) >= 2, logical(1))]
  if (!length(fe)) next

  ## 5) 秩亏/共线检查：逐个去掉导致秩亏的变量
  repeat {
    mm <- model.matrix(as.formula(paste("~", paste(fe, collapse = "+"))), data = gm)
    if (qr(mm)$rank == ncol(mm) || length(fe) == 1) break
    fe <- fe[-length(fe)]        # 或按你的优先级决定丢弃哪个
  }
  mm <- model.matrix(as.formula(paste("~", paste(fe, collapse = "+"))), data = gm)
  if (qr(mm)$rank < ncol(mm)) next

  ps <- phyloseq(otu_table(as.matrix(sub), taxa_are_rows = TRUE), sample_data(gm))

  fit <- tryCatch(
    ancombc2(data = ps, tax_level = NULL,
             fix_formula  = paste(fe, collapse = " + "),
             rand_formula = "(1 | BioProject)",
             p_adj_method = "BH",
             prv_cut = 0,         
             lib_cut = 0, s0_perc = 0.05, group = NULL,
             struc_zero = FALSE, neg_lb = FALSE,
             alpha = MAX_SIGNIFICANCE, n_cl = ANCOMBC_CORES, verbose = TRUE,
             global = FALSE, pairwise = FALSE, dunnet = FALSE, trend = FALSE,
             iter_control = list(tol = 1e-2, max_iter = 20, verbose = FALSE),
             em_control   = list(tol = 1e-5, max_iter = 100),
             lme_control  = lme4::lmerControl(),
             mdfdr_control = list(fwer_ctrl_method = "holm", B = 100)),
    error = function(e) { warning(sprintf("Geography %s failed: %s", g, conditionMessage(e))); NULL })

  if (is.null(fit)) next
  r <- tidy_ancombc2(fit$res, fe); r$Geography <- g
  geo_results[[g]] <- r
}

Checking the input data type ...

The input data is of type: phyloseq

PASS

Checking the sample metadata ...

The specified variables in the formula: Age_Group, Sex, Region_custom

The available variables in the sample metadata: BioProject, SampleID, subject_id, chim_perc, asv_count, total_reads, sample_cr_mapping_rate, study_cr_mapping_rate, Observed, Shannon, Simpson, InvSimpson, Chao1, age, age_info, bmi, sex, platform, sequencing_region, pcr_primers, country, site, site_group, disease_study, status, title, public_time, matched_samples, is_time_series, age_cleaned, bmi_cleaned, Age_Group, age_info_cleaned, bmi_cleaned_group, Sex, platform_cleaned, Region_custom, country_cleaned, Geography, Site_Cleaned, site_group_cleaned, disease_study_cleaned, Status_Cleaned, asv_le_30, study_asv_lt30_high, chim_gt_25, study_chim_gt25_high, sample_cr_mapping_low, study_cr_mapping_low

PASS

Checking other arguments ...

PASS

Obtaining initial estimates ...

Estimating sample-specific biases ...


In [39]:
combined <- bind_rows(geo_results, .id = "Geography")

In [40]:
length(geo_results)

[1] 4

In [41]:
str(combined)
head(combined, 3)

'data.frame':	2590 obs. of  13 variables:
 $ feature        : chr  "AEES01000073" "AWVT01" "Abiotrophia" "Acinetobacter" ...
 $ metadata       : chr  "Age_Group" "Age_Group" "Age_Group" "Age_Group" ...
 $ value          : chr  "12-18 years" "12-18 years" "12-18 years" "12-18 years" ...
 $ term           : chr  "Age_Group12-18 years" "Age_Group12-18 years" "Age_Group12-18 years" "Age_Group12-18 years" ...
 $ coef           : num  0 0 0 0.13 2.21 ...
 $ stderr         : num  0.499 0.499 0.499 1.735 1.169 ...
 $ stat           : num  0 0 0 0.0751 1.8883 ...
 $ pval_individual: num  1 1 1 0.9403 0.0591 ...
 $ qval_individual: num  1 1 1 1 0.716 ...
 $ diff_abundant  : logi  FALSE FALSE FALSE FALSE FALSE FALSE ...
 $ passed_ss      : logi  TRUE TRUE TRUE FALSE FALSE FALSE ...
 $ model          : chr  "ANCOM-BC2" "ANCOM-BC2" "ANCOM-BC2" "ANCOM-BC2" ...
 $ Geography      : chr  "Asia" "Asia" "Asia" "Asia" ...


,feature,metadata,value,term,coef,stderr,stat,pval_individual,qval_individual,diff_abundant,passed_ss,model,Geography
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<chr>,<chr>
1,AEES01000073,Age_Group,12-18 years,Age_Group12-18 years,0,0.4993387,0,1,1,FALSE,TRUE,ANCOM-BC2,Asia
2,AWVT01,Age_Group,12-18 years,Age_Group12-18 years,0,0.4993387,0,1,1,FALSE,TRUE,ANCOM-BC2,Asia
3,Abiotrophia,Age_Group,12-18 years,Age_Group12-18 years,0,0.4993387,0,1,1,FALSE,TRUE,ANCOM-BC2,Asia


In [42]:
reg <- combined %>%
  filter(metadata == "Region_custom") %>%
  mutate(
    Region = value,
    signed = sign(coef) * -log10(qval_individual),
    sig    = !is.na(qval_individual) & qval_individual < 0.1 & passed_ss
  )

table(reg$Region, reg$Geography)
table(reg$sig, reg$passed_ss)

       
        Asia North America Oceania South America
  V1       0             0       0            72
  V1-V2  109            89       0             0
  V1-V3    0            89       0             0
  V3     109             0       0             0
  V3-V4  109            89     107             0
  V4-V5  109             0       0             0
  V6-V8    0             0     107             0

       
        FALSE TRUE
  FALSE   110  867
  TRUE      0   12

In [43]:
with(reg, table(diff_abundant, qval_individual < 0.1, passed_ss)) 

, , passed_ss = FALSE

             
diff_abundant FALSE TRUE
        FALSE    89    0
        TRUE     11   10

, , passed_ss = TRUE

             
diff_abundant FALSE TRUE
        FALSE   862    0
        TRUE      5   12


In [44]:
keep <- reg %>%
  filter(sig) %>%
  group_by(feature, Region) %>%
  summarise(n_sig = n(), n_dir = n_distinct(sign(coef)), .groups = "drop") %>%
  filter(n_sig >= 2, n_dir == 1) %>%
  select(feature, Region)

plot_df <- reg %>% semi_join(keep, by = c("feature", "Region"))

nrow(keep); n_distinct(plot_df$feature)

[1] 0

[1] 0

# LinDA

In [46]:
tidy_linda <- function(fit, fe) {
  out <- fit$output
  if (!length(out)) return(NULL)
  do.call(rbind, lapply(names(out), function(cf) {
    d <- out[[cf]]
    data.frame(
      taxon    = rownames(d),
      term     = cf,
      variable = {                       # 从系数名反推是哪个变量
        hit <- fe[vapply(fe, function(v) startsWith(cf, v), logical(1))]
        if (length(hit)) hit[which.max(nchar(hit))] else NA_character_
      },
      lfc      = d$log2FoldChange,       # 注意：log2 尺度
      se       = d$lfcSE,
      stat     = d$stat,
      p        = d$pvalue,
      q        = d$padj,
      df       = d$df,
      baseMean = d$baseMean,
      row.names = NULL, stringsAsFactors = FALSE
    )
  }))
}

geo_results <- list()
for (g in levels(droplevels(sal$Geography))) {
  gm   <- sal[sal$Geography == g, , drop = FALSE]
  vars <- c("Age_Group","Sex","Region_custom","BioProject")

  ## 1) 删协变量含 NA 的样本
  gm <- gm[complete.cases(gm[, vars]), , drop = FALSE]

  ## 2) taxa 过滤，再删零计数样本
  sub <- abund[, rownames(gm), drop = FALSE]
  sub <- sub[rowSums(sub) > 0, , drop = FALSE]
  sub <- sub[rowSums(sub > 0) >= 2, , drop = FALSE]
  sub <- sub[rowMeans(sub > 0) >= MIN_PREVALENCE, , drop = FALSE]
  if (nrow(sub) < 5) next
  gm  <- gm[colSums(sub) > 0, , drop = FALSE]
  sub <- sub[, rownames(gm), drop = FALSE]

  ## 3) droplevels
  for (v in vars) gm[[v]] <- droplevels(gm[[v]])
  if (nlevels(gm$BioProject) < 2) next
  if (nlevels(gm$Region_custom) >= 2 && "V4" %in% levels(gm$Region_custom))
    gm$Region_custom <- relevel(gm$Region_custom, "V4")

  ## 4) 只保留 >=2 水平的固定效应
  fe <- c("Age_Group","Sex","Region_custom")
  fe <- fe[vapply(fe, function(x) nlevels(gm[[x]]) >= 2, logical(1))]
  if (!length(fe)) next

  ## 5) 秩亏/共线检查
  repeat {
    mm <- model.matrix(as.formula(paste("~", paste(fe, collapse = "+"))), data = gm)
    if (qr(mm)$rank == ncol(mm) || length(fe) == 1) break
    fe <- fe[-length(fe)]
  }
  mm <- model.matrix(as.formula(paste("~", paste(fe, collapse = "+"))), data = gm)
  if (qr(mm)$rank < ncol(mm)) next

  ## 6) LinDA：随机效应写进 formula
  form <- paste("~", paste(fe, collapse = " + "), "+ (1 | BioProject)")

  fit <- tryCatch(
    linda(feature.dat      = as.matrix(sub),
          meta.dat         = as.data.frame(gm),
          formula          = form,
          feature.dat.type = "count",
          prev.filter      = 0,        # 已在外面筛过
          mean.abund.filter= 0,
          max.abund.filter = 0,
          is.winsor        = TRUE,
          outlier.pct      = 0.03,
          adaptive         = TRUE,
          zero.handling    = "pseudo-count",
          pseudo.cnt       = 0.5,
          corr.cut         = 0.1,
          p.adj.method     = "BH",
          alpha            = MAX_SIGNIFICANCE,
          n.cores          = ANCOMBC_CORES,
          verbose          = TRUE),
    error = function(e) { warning(sprintf("Geography %s failed: %s", g, conditionMessage(e))); NULL })

  if (is.null(fit)) next
  r <- tidy_linda(fit, fe)
  if (is.null(r)) next
  r$Geography <- g
  geo_results[[g]] <- r
}

combined <- do.call(rbind, geo_results)

Registered S3 method overwritten by 'rmutil':
  method         from
  print.response httr



0  features are filtered!
The filtered data has  2550  samples and  109  features will be tested!
Pseudo-count approach is used.
Fit linear mixed effects models ...
Completed.
0  features are filtered!
The filtered data has  2652  samples and  91  features will be tested!
Imputation approach is used.
Fit linear mixed effects models ...
Completed.
0  features are filtered!
The filtered data has  7784  samples and  89  features will be tested!
Pseudo-count approach is used.
Fit linear mixed effects models ...
Completed.
0  features are filtered!
The filtered data has  139  samples and  107  features will be tested!


boundary (singular) fit: see help('isSingular')



Imputation approach is used.
Fit linear mixed effects models ...
Completed.
0  features are filtered!
The filtered data has  102  samples and  72  features will be tested!
Pseudo-count approach is used.
Fit linear mixed effects models ...
Completed.


In [53]:
combined <- geo_all

In [54]:
str(combined)
head(combined, 3)

'data.frame':	3409 obs. of  11 variables:
 $ taxon    : chr  "AEES01000073" "AWVT01" "Abiotrophia" "Acinetobacter" ...
 $ term     : chr  "Age_Group12-18 years" "Age_Group12-18 years" "Age_Group12-18 years" "Age_Group12-18 years" ...
 $ variable : chr  "Age_Group" "Age_Group" "Age_Group" "Age_Group" ...
 $ lfc      : num  0.565 -0.572 4.925 -6.638 5.517 ...
 $ se       : num  1.25 1.35 2.8 1.75 2.22 ...
 $ stat     : num  0.451 -0.423 1.759 -3.798 2.491 ...
 $ p        : num  0.651779 0.672511 0.078788 0.000149 0.012825 ...
 $ q        : num  0.86562 0.86562 0.23855 0.00205 0.06078 ...
 $ df       : num  2155 1893 1741 2315 1995 ...
 $ baseMean : num  3.99 12.14 15.86 2357.53 40.19 ...
 $ Geography: chr  "Asia" "Asia" "Asia" "Asia" ...


,taxon,term,variable,lfc,se,stat,p,q,df,baseMean,Geography
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Asia.1,AEES01000073,Age_Group12-18 years,Age_Group,0.5653790,1.252625,0.4513553,0.65177890,0.8656186,2155.012,3.987608,Asia
Asia.2,AWVT01,Age_Group12-18 years,Age_Group,-0.5723184,1.353734,-0.4227701,0.67251100,0.8656186,1893.340,12.136895,Asia
Asia.3,Abiotrophia,Age_Group12-18 years,Age_Group,4.9251457,2.800293,1.7587967,0.07878776,0.2385518,1740.539,15.862764,Asia


In [57]:
reg <- combined %>%
  filter(variable == "Region_custom") %>%
  mutate(
    Region = term,
    signed = lfc,
    sig    = !is.na(q) & q < 0.1
  )

table(reg$Region, reg$Geography)
table(reg$sig)

                    
                     Asia Europe North America Oceania South America
  Region_customV1       0      0             0       0            72
  Region_customV1-V2  109     91            89       0             0
  Region_customV1-V3    0      0            89       0             0
  Region_customV3     109     91             0       0             0
  Region_customV3-V4  109     91            89     107             0
  Region_customV4-V5  109      0             0       0             0
  Region_customV6-V8    0      0             0     107             0


FALSE  TRUE 
 1132   130 

In [60]:
head(reg)

,taxon,term,variable,lfc,se,stat,p,q,df,baseMean,Geography,Region,signed,sig
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<lgl>
Asia.655,AEES01000073,Region_customV1-V2,Region_custom,1.2316449,0.6651464,1.8516900,0.07412681,0.2331207,29.43191,3.987608,Asia,Region_customV1-V2,1.2316449,FALSE
Asia.656,AWVT01,Region_customV1-V2,Region_custom,0.6106698,0.4916546,1.2420707,0.22413008,0.4550309,29.09358,12.136895,Asia,Region_customV1-V2,0.6106698,FALSE
Asia.657,Abiotrophia,Region_customV1-V2,Region_custom,0.9645240,0.7752233,1.2441887,0.22572781,0.4550309,23.44129,15.862764,Asia,Region_customV1-V2,0.9645240,FALSE
Asia.658,Acinetobacter,Region_customV1-V2,Region_custom,0.1263487,1.1741827,0.1076057,0.91510679,0.9536695,26.90894,2357.531360,Asia,Region_customV1-V2,0.1263487,FALSE
Asia.659,Actinomyces,Region_customV1-V2,Region_custom,-0.1813404,1.0249292,-0.1769297,0.86093325,0.9351397,26.02801,40.194293,Asia,Region_customV1-V2,-0.1813404,FALSE
Asia.660,Aggregatibacter_736122,Region_customV1-V2,Region_custom,-0.6685156,1.3198141,-0.5065226,0.61632443,0.7774066,28.96203,5.707387,Asia,Region_customV1-V2,-0.6685156,FALSE


In [64]:
keep <- reg %>%
  filter(sig) %>%
  group_by(taxon, Region) %>%
  summarise(n_sig = n(), n_dir = n_distinct(sign(lfc)), .groups = "drop") %>%
  filter(n_sig >= 2, n_dir == 1) %>%
  select(taxon, Region)

plot_df <- reg %>% semi_join(keep, by = c("taxon", "Region"))

nrow(keep); n_distinct(plot_df$taxon)

[1] 1

[1] 1